# Library

In [62]:
pip install torch_geometric

In [63]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv, Linear

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


# Path

In [64]:
BASE = Path(".")
NB2 = BASE / "nb2"
NB3 = BASE / "nb3"
SLGAD = BASE / "nb4"
OUT = BASE / "nb5"
OUT.mkdir(parents=True, exist_ok=True)

# Inputs
HISN_PARQUET      = NB2 / "hisn_final_with_rid.parquet"
NODES_CSV         = NB3 / "nodes.csv"
EDGES_CSV         = NB3 / "edges.csv"
SPAMMER_USERS_CSV = NB3 / "spammer_users.csv"

# SL-GAD FILES (fixed paths)
SLGAD_EMB_NPY   = SLGAD / "slgad_review_embeddings.npy"
SLGAD_SCORE_NPY = SLGAD / "slgad_review_scores.npy"

# Outputs
PSEUDO_LABELS_CSV = OUT / "reviews_pseudo_labels.csv"
FINAL_PRED_CSV    = OUT / "final_predictions.csv"
MODEL_STATE_PATH  = OUT / "hgnn_model.pt"

# Load Files

## HISN

In [65]:
df_hisn = pd.read_parquet(HISN_PARQUET).reset_index(drop=True)

In [66]:
df_hisn.shape

(6461, 35)

In [67]:
df_hisn.head(2)

,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,summary,...,user_degree,item_degree,local_density,rating_scaled,review_length_scaled,log_helpful_scaled,verified_int_scaled,dup_count_scaled,orig_index,review_id
0,5,fancy pumpkin headband,Purchased for my sister to use on Halloween......,B071HMN7K8,B071HMN7K8,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2021-10-27 04:30:00.822,0,True,,...,6,9,0.012836,0.701577,0.030073,-0.435584,0.393727,-0.146009,7,r_0
1,5,abalone swirl necklace,This is a large swirl abalone necklace... it i...,B01LZUP2XY,B01LZUP2XY,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2020-01-12 17:42:18.354,2,True,,...,6,4,0.028881,0.701577,1.879858,1.489888,0.393727,-0.146009,8,r_1


In [68]:
df_hisn.columns.tolist()

['rating',
 'title',
 'text',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase',
 'summary',
 'reviewText',
 'full_text_norm',
 'full_text',
 'review_length',
 'dup_count',
 'is_duplicate',
 'log_helpful',
 'verified_int',
 'day',
 'hour',
 'sin_day',
 'cos_day',
 'sin_hour',
 'cos_hour',
 'date',
 'user_degree',
 'item_degree',
 'local_density',
 'rating_scaled',
 'review_length_scaled',
 'log_helpful_scaled',
 'verified_int_scaled',
 'dup_count_scaled',
 'orig_index',
 'review_id']

## Load Nodes and Edges

In [69]:
nodes = pd.read_csv(NODES_CSV)
edges = pd.read_csv(EDGES_CSV)

In [70]:
print("Node types:\n", nodes["node_type"].value_counts())

Node types:
 node_type
review     6461
product    4287
user       1449
Name: count, dtype: int64


In [71]:
nodes.head(3)

,node_id,node_type,user_n_reviews_user,user_avg_rating_user,user_std_rating_user,user_frac_verified_user,user_avg_len_user,user_dup_ratio_user,user_max_reviews_per_day,user_day_entropy,user_avg_text_sim,item_n_reviews_item,item_avg_rating_item,item_std_rating_item,item_dup_ratio_item,item_n_unique_users
0,AFZUK3MTBIBEDQOPAK3OATUOUKLA,user,6.0,4.666667,0.816497,1.000000,113.166667,0.0,1.0,1.791759,0.373616,NaN,NaN,NaN,NaN,NaN
1,AFA26DYXVLJYTYZ3KET77GE27N2A,user,4.0,4.250000,1.500000,1.000000,57.500000,0.0,1.0,1.386294,0.317828,NaN,NaN,NaN,NaN,NaN
2,AHJQPUQLSQZE6LMIUMY7WNRXCQQQ,user,29.0,3.758621,1.243703,0.034483,134.862069,0.0,2.0,3.271689,0.482127,NaN,NaN,NaN,NaN,NaN


In [72]:
print("Edge types:\n", edges["edge_type"].value_counts())

Edge types:
 edge_type
user-writes-review        6461
review-about-item         6461
review-written-by-user    6461
item-has-review           6461
Name: count, dtype: int64


In [73]:
edges.head(3)

,src,dst,edge_type
0,AFZUK3MTBIBEDQOPAK3OATUOUKLA,r_0,user-writes-review
1,AFZUK3MTBIBEDQOPAK3OATUOUKLA,r_1,user-writes-review
2,AFZUK3MTBIBEDQOPAK3OATUOUKLA,r_2,user-writes-review


## Load SL-GAD Outputs

In [74]:
review_emb = np.load(SLGAD_EMB_NPY)
anomaly_scores = np.load(SLGAD_SCORE_NPY)

In [75]:
review_emb.shape, anomaly_scores.shape

((6461, 64), (6461,))

Here we also assign the anomaly scores from SL-GAD to our HISN dataframe/

In [76]:
df_hisn["anomaly"] = anomaly_scores
print("\nAnomaly stats:")
print(df_hisn["anomaly"].describe())


Anomaly stats:
count    6461.000000
mean        0.044751
std         0.015355
min         0.010241
25%         0.037098
50%         0.043132
75%         0.050233
max         0.203487
Name: anomaly, dtype: float64


## Load SGDCTH spammer users

In [77]:
spammer_users = pd.read_csv(SPAMMER_USERS_CSV)
display(spammer_users.head(3)), spammer_users.info()

,user_id,cluster,sgdcth_score
0,AE4QXRRYHSFY6GNM2IAV7E5W4AYA,1,1.0
1,AE6M4O3IVW56BCMCYJSZNCSQU5IQ,1,1.0
2,AE6OCD3CYMEX6GXPEBB5VMM3ZL2Q,1,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   user_id       40 non-null     object 
 1   cluster       40 non-null     int64  
 2   sgdcth_score  40 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ KB


(None, None)

In [78]:
spam_user_set = set(spammer_users["user_id"].unique())
len(spam_user_set)

40

Here we can see the 2390 users that are flagged as suspicious by our SGDCTH, lets assign it too our df_hisn too

In [79]:
df_hisn["user_suspicious"] = df_hisn["user_id"].isin(spam_user_set).astype(int)

In [80]:
print("\nuser_suspicious value counts:")
print(df_hisn["user_suspicious"].value_counts())


user_suspicious value counts:
user_suspicious
0    6371
1      90
Name: count, dtype: int64


In [81]:
df_hisn[["user_id","anomaly","user_suspicious"]].sort_values(by="anomaly", ascending=False).head(10)

,user_id,anomaly,user_suspicious
2193,AFSYS2MOJIZGIFAIOKYIHK7AVZOA,0.203487,0
4823,AFJ5JDYII5AD5SR54XS5CNOBN2CQ,0.200803,0
6170,AFE3LHXPUNZKLRS6ARSOCYWT5GZQ,0.197826,0
5538,AGRCB7G6HJOJZ3I7MMORJS6TDILQ,0.197793,0
5537,AGRCB7G6HJOJZ3I7MMORJS6TDILQ,0.197245,0
4356,AG6TVMVNIAFWKAU3JLY73Z6JCXKA,0.197171,0
3823,AEDR47E6Z2KCPXKEQV2SRKKIULCA,0.197149,0
2041,AGJ2RUDVDLEB46GV7E7SGBDP7MOQ,0.196789,0
4913,AEWLZOQEM5JQDHYFJQEAIBJSK3TA,0.196758,0
6221,AE4QXRRYHSFY6GNM2IAV7E5W4AYA,0.196690,1


# Pseudo Label Generation

We are gonna create some pseudo label with the merge of SLGAD and SGDCTH technique with some of this approach :<br>

1. SLGAD <br>
From the anomaly score we gonna take the high and low cutoff using n% of alpha-beta

2. SGDCTH   <br>
We gonna take the label of user suspicious from this process.

3. Labeling rule <br>
if user_suspicious = 1 and >= hi cut == fake <br>
if user_suspicious = 0 and <= lo cut == genuine <br>
otherwise = -1 --> unlabeled

<br>
By this approach we gonna generate some of the pseudo labels that are having a high precision.

In [82]:
alpha = 0.05
beta = 0.05

In [83]:
hi_cut = np.percentile(df_hisn["anomaly"], 100*(1-alpha))
lo_cut = np.percentile(df_hisn["anomaly"], 100*(beta))

print(f"High anomaly cutoff (top {alpha*100:.1f}%): {hi_cut:.6f}")
print(f"Low anomaly cutoff (bottom {beta*100:.1f}%): {lo_cut:.6f}")

High anomaly cutoff (top 5.0%): 0.063996
Low anomaly cutoff (bottom 5.0%): 0.026965


In [84]:
# This is the conditional function to assign the pseudolabels

def pseudo_label_row(row):
    # High-anomaly + suspicious user = pseudo-fake
    if row["anomaly"] >= hi_cut and row["user_suspicious"] == 1:
        return 1  # fake
    # Low-anomaly + non-suspicious user = pseudo-genuine
    if row["anomaly"] <= lo_cut and row["user_suspicious"] == 0:
        return 0  # genuine
    # Otherwise unlabeled
    return -1

In [85]:
df_hisn["pseudo_label"] = df_hisn.apply(pseudo_label_row, axis=1)

In [86]:
print("\nPseudo-label counts (-1 = unlabeled):")
df_hisn["pseudo_label"].value_counts()


Pseudo-label counts (-1 = unlabeled):


,count
pseudo_label,
-1,6157
0,298
1,6


In [87]:
print("\nPseudo-label vs user_suspicious cross-tab:")
pd.crosstab(df_hisn["pseudo_label"], df_hisn["user_suspicious"])


Pseudo-label vs user_suspicious cross-tab:


user_suspicious,0,1
pseudo_label,,
-1,6073,84
0,298,0
1,0,6


In [88]:
df_hisn.to_csv(PSEUDO_LABELS_CSV, index=False)
print("\n✅ Pseudo-labels saved to", PSEUDO_LABELS_CSV)


✅ Pseudo-labels saved to nb5/reviews_pseudo_labels.csv


# Build HeteroData (Graph) + Add Reverse Edges

1. We gonna make some nodes:
- data["review"].x from the SL-GAD embeddings
- data["user"].x from the user_ columns* in the nodes table
- data["product"].x from the item_ columns* in the nodes table

2. Set ID Mappings
- review_id → review index
- user_id → user index
- asin → product index

3. Add Edges
- "user-reviews-product" → (user, "writes", review)
- "review-about-product" → (review, "about", product)

In [89]:
from torch_geometric.data import HeteroData

In [90]:
def build_heterodata_for_hgnn(df_h, nodes_df, edges_df, review_emb_np):
    data = HeteroData()

    # ---- REVIEW NODES (SL-GAD embeddings) ----
    data["review"].x = torch.tensor(review_emb_np, dtype=torch.float)

    # ---- USER NODES ----
    user_nodes = nodes_df[nodes_df["node_type"] == "user"].copy()
    user_nodes["node_id"] = user_nodes["node_id"].astype(str)
    user_feat_cols = [c for c in user_nodes.columns if c.startswith("user_")]
    user_x = user_nodes[user_feat_cols].fillna(0.0).to_numpy()
    data["user"].x = torch.tensor(user_x, dtype=torch.float)

    # ---- PRODUCT NODES ----
    prod_nodes = nodes_df[nodes_df["node_type"] == "product"].copy()
    prod_nodes["node_id"] = prod_nodes["node_id"].astype(str)
    prod_feat_cols = [c for c in prod_nodes.columns if c.startswith("item_")]
    prod_x = prod_nodes[prod_feat_cols].fillna(0.0).to_numpy()
    data["product"].x = torch.tensor(prod_x, dtype=torch.float)

    # ---- ID MAPPINGS ----
    review_ids  = df_h["review_id"].astype(str).tolist()
    user_ids    = user_nodes["node_id"].tolist()
    product_ids = prod_nodes["node_id"].tolist()

    rev2idx = {r: i for i, r in enumerate(review_ids)}
    uid2idx = {u: i for i, u in enumerate(user_ids)}
    pid2idx = {p: i for i, p in enumerate(product_ids)}

    # ---- EDGES: user -> review (writes) ----
    e_ur = edges_df[edges_df["edge_type"] == "user-writes-review"].copy()
    e_ur["src_idx"] = e_ur["src"].astype(str).map(uid2idx)
    e_ur["dst_idx"] = e_ur["dst"].astype(str).map(rev2idx)
    e_ur = e_ur.dropna(subset=["src_idx", "dst_idx"])

    ur_src = e_ur["src_idx"].astype(int).to_numpy()
    ur_dst = e_ur["dst_idx"].astype(int).to_numpy()
    data["user", "writes", "review"].edge_index = torch.tensor(
        [ur_src, ur_dst], dtype=torch.long
    )

    # ---- EDGES: review -> product (about) ----
    e_rp = edges_df[edges_df["edge_type"] == "review-about-item"].copy()
    e_rp["src_idx"] = e_rp["src"].astype(str).map(rev2idx)
    e_rp["dst_idx"] = e_rp["dst"].astype(str).map(pid2idx)
    e_rp = e_rp.dropna(subset=["src_idx", "dst_idx"])

    rp_src = e_rp["src_idx"].astype(int).to_numpy()
    rp_dst = e_rp["dst_idx"].astype(int).to_numpy()
    data["review", "about", "product"].edge_index = torch.tensor(
        [rp_src, rp_dst], dtype=torch.long
    )

    # ---- EDGES: review -> user (written_by, reverse of user->review) ----
    e_ru = edges_df[edges_df["edge_type"] == "review-written-by-user"].copy()
    # despite the name, in NB2 this is review_id -> user_id
    e_ru["src_idx"] = e_ru["src"].astype(str).map(rev2idx)
    e_ru["dst_idx"] = e_ru["dst"].astype(str).map(uid2idx)
    e_ru = e_ru.dropna(subset=["src_idx", "dst_idx"])

    if len(e_ru) > 0:
        ru_src = e_ru["src_idx"].astype(int).to_numpy()
        ru_dst = e_ru["dst_idx"].astype(int).to_numpy()
        data["review", "written_by", "user"].edge_index = torch.tensor(
            [ru_src, ru_dst], dtype=torch.long
        )

    # ---- EDGES: product -> review (has_review, reverse of review->product) ----
    e_pr = edges_df[edges_df["edge_type"] == "item-has-review"].copy()
    e_pr["src_idx"] = e_pr["src"].astype(str).map(pid2idx)
    e_pr["dst_idx"] = e_pr["dst"].astype(str).map(rev2idx)
    e_pr = e_pr.dropna(subset=["src_idx", "dst_idx"])

    if len(e_pr) > 0:
        pr_src = e_pr["src_idx"].astype(int).to_numpy()
        pr_dst = e_pr["dst_idx"].astype(int).to_numpy()
        data["product", "has_review", "review"].edge_index = torch.tensor(
            [pr_src, pr_dst], dtype=torch.long
        )

    return data

In [91]:
data = build_heterodata_for_hgnn(df_hisn, nodes, edges, review_emb)
data = data.to(DEVICE)

In [92]:
print("\n=== HeteroData SUMMARY ===")
print(data)
print("\nNode types:", data.node_types)
print("Edge types:", data.edge_types)
print("Metadata:", data.metadata())


=== HeteroData SUMMARY ===
HeteroData(
  review={ x=[6461, 64] },
  user={ x=[1449, 9] },
  product={ x=[4287, 5] },
  (user, writes, review)={ edge_index=[2, 6461] },
  (review, about, product)={ edge_index=[2, 6461] },
  (review, written_by, user)={ edge_index=[2, 6461] },
  (product, has_review, review)={ edge_index=[2, 6461] }
)

Node types: ['review', 'user', 'product']
Edge types: [('user', 'writes', 'review'), ('review', 'about', 'product'), ('review', 'written_by', 'user'), ('product', 'has_review', 'review')]
Metadata: (['review', 'user', 'product'], [('user', 'writes', 'review'), ('review', 'about', 'product'), ('review', 'written_by', 'user'), ('product', 'has_review', 'review')])


In [93]:
print("\nNode feature shapes:")
print("review:", data["review"].x.shape)
print("user:", data["user"].x.shape)
print("product:", data["product"].x.shape)


Node feature shapes:
review: torch.Size([6461, 64])
user: torch.Size([1449, 9])
product: torch.Size([4287, 5])


In [94]:
print("\nEdge index shapes:")
for et, ei in data.edge_index_dict.items():
    print(et, ":", ei.shape)


Edge index shapes:
('user', 'writes', 'review') : torch.Size([2, 6461])
('review', 'about', 'product') : torch.Size([2, 6461])
('review', 'written_by', 'user') : torch.Size([2, 6461])
('product', 'has_review', 'review') : torch.Size([2, 6461])


# Model Definition of HGNN and Regularization

In [95]:
from torch_geometric.nn import HGTConv

class ReviewHGNN(nn.Module):
    def __init__(self, data, hidden=64, dropout_p=0.3):
        super().__init__()

        review_dim  = data["review"].num_features
        user_dim    = data["user"].num_features
        product_dim = data["product"].num_features

        print(f"Review dim:  {review_dim}")
        print(f"User dim:    {user_dim}")
        print(f"Product dim: {product_dim}")
        print(f"Hidden dim:  {hidden}")

        # Input projections per node type
        self.in_review  = Linear(review_dim, hidden)
        self.in_user    = Linear(user_dim, hidden)
        self.in_product = Linear(product_dim, hidden)

        # Heterogeneous graph transformer
        self.hgt = HGTConv(
            in_channels=hidden,
            out_channels=hidden,
            metadata=data.metadata(),
            heads=2
        )

        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, data):
        x_dict = {
            "review":  self.in_review(data["review"].x),
            "user":    self.in_user(data["user"].x),
            "product": self.in_product(data["product"].x),
        }

        x_dict = self.hgt(x_dict, data.edge_index_dict)

        # Take review embeddings, apply dropout, classify
        review_repr = self.dropout(x_dict["review"])
        logits = self.classifier(review_repr)
        return logits

In [96]:
metadata = data.metadata()
model = ReviewHGNN(data, hidden = 64, dropout_p=0.3).to(DEVICE)

Review dim:  64
User dim:    9
Product dim: 5
Hidden dim:  64


In [97]:
model

ReviewHGNN(
  (in_review): Linear(64, 64, bias=True)
  (in_user): Linear(9, 64, bias=True)
  (in_product): Linear(5, 64, bias=True)
  (hgt): HGTConv(-1, 64, heads=2)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)

# Training Setup

We gonna set
- Optimizer using Adam
- using loss with Cross Entropy
- labels from pseudo labels
- train epoch

## Optimzier

In [98]:
optimizer = torch.optim.Adam(model.parameters(), lr = 2e-4, weight_decay=1e-4)
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0002
    maximize: False
    weight_decay: 0.0001
)

## Labels

In [99]:
labels_np = df_hisn["pseudo_label"].to_numpy()
labels  = torch.tensor(labels_np, dtype=torch.long).to(DEVICE)
labels

tensor([-1, -1, -1,  ..., -1, -1, -1], device='cuda:0')

In [100]:
mask_labeled = labels != -1
print("Total reviews:", len(labels))
print("Initially labeled samples:", int(mask_labeled.sum()))

Total reviews: 6461
Initially labeled samples: 304


In [101]:
initial_labeled = labels[mask_labeled].detach().cpu().numpy()
unique_init, counts_init = np.unique(initial_labeled, return_counts=True)

result = {int(k): int(v) for k, v in zip(unique_init, counts_init)}
print("Initial labeled class distribution:", result)

Initial labeled class distribution: {0: 298, 1: 6}


## Loss

In [102]:
loss_fn = nn.CrossEntropyLoss()

## Traning func

In [103]:
def train_epoch(mask):
    model.train()
    optimizer.zero_grad()
    logits = model(data)
    loss = loss_fn(logits[mask], labels[mask])
    loss.backward()
    optimizer.step()
    return float(loss.item())

# Round 0

Round 0 serves as a warm-up stage. By training only on the high-precision pseudo-labels obtained from the intersection of SL-GAD and SGDCTH, the model avoids noisy supervision and learns a stable initial embedding space. This provides ReviewHGNN with a reliable representation before progressively incorporating lower-confidence pseudo-labels in later rounds.

In [104]:
print("=== ROUND 0: Train on high-precision pseudo-labels ===")
epochs_round0 = 200
for epoch in range(epochs_round0):
    loss = train_epoch(mask_labeled)
    if epoch % 20 == 0 or epoch == epochs_round0 - 1:
        print(f"[Round 0] Epoch {epoch:03d} | loss = {loss:.4f}")

=== ROUND 0: Train on high-precision pseudo-labels ===
[Round 0] Epoch 000 | loss = 0.6324
[Round 0] Epoch 020 | loss = 0.1706
[Round 0] Epoch 040 | loss = 0.0971
[Round 0] Epoch 060 | loss = 0.0977
[Round 0] Epoch 080 | loss = 0.0822
[Round 0] Epoch 100 | loss = 0.0769
[Round 0] Epoch 120 | loss = 0.0766
[Round 0] Epoch 140 | loss = 0.0658
[Round 0] Epoch 160 | loss = 0.0577
[Round 0] Epoch 180 | loss = 0.0508
[Round 0] Epoch 199 | loss = 0.0376


# Curriculum Self-Training Loop

Curriculum Self-Training is a learning strategy designed to gradually expand the set of labeled samples used for training a model. Instead of immediately trusting all pseudo-labels predicted by the model, this approach introduces new training samples step-by-step, starting from the most confident ones and progressively moving toward more difficult cases.<br>
<br>
This ensures that the model learns in a stable, controlled, and noise-resistant manner.<br><br>
In semi-supervised graph learning, the majority of nodes are unlabeled. Directly training the model using all pseudo-labels at once may introduce large amounts of noise and cause error propagation.

To avoid this, Curriculum Self-Training enforces a learning schedule:

1. Begin with high-precision pseudo-labels
2. Gradually increase the number of pseudo-labeled samples
3. Slowly decrease the confidence threshold in each round
4. Allow the model to learn from easier samples first
5. Incorporate harder (lower-confidence) samples later

This mimics how humans learn — starting from clear examples before dealing with ambiguous ones.

***Curriculum Loop (Rounds 1–10)***

The curriculum consists of multiple rounds. Each round performs the following steps:

1. **Model Inference**

The model predicts the class probabilities for all nodes.<br>
For each node:
- pred[i] = predicted class
- conf[i] = highest class probability

2. **Confidence Thresholding** <br>
Each round has a predefined confidence threshold, only unlabeled data whose prediction confidence is same or higher than threshold will be labeled and accepted as new pseudo-labeled samples

This ensures:
- Round 1 → only extremely obvious samples added
- Round 10 → lower-confidence samples allowed
Thus, the training set grows gradually and safely.

3. **Updating Labels**

Accepted nodes are assigned the predicted class, permanently becoming part of the labeled set.

4. **Fine-Tuning**

The model is retrained for several epochs using the updated labeled set.
This allows the model to incorporate new information without forgetting previous knowledge.

5. **Early Stopping**

If no new nodes exceed the threshold in a given round, the curriculum loop terminates early.

In [105]:
# Absolute threshold schedule
threshold_schedule = {
    1: 0.99, 2: 0.97, 3: 0.95, 4: 0.93, 5: 0.91,
    6: 0.89, 7: 0.87, 8: 0.85, 9: 0.83, 10: 0.80
}

In [106]:
num_rounds = 10

In [107]:
for r in range(1, num_rounds + 1):
    thr = threshold_schedule[r]
    print(f"\n\n==============================")
    print(f"ROUND {r} — Confidence Threshold = {thr:.4f}")
    print(f"==============================\n")

    # --- Inference ---
    model.eval()
    with torch.no_grad():
        logits = model(data)
        probs = torch.softmax(logits, dim=1)
        conf, pred = probs.max(dim=1)

    # Confidence stats
    conf_np = conf.detach().cpu().numpy()
    print(" Confidence Statistics")
    print(f"   • min   : {conf_np.min():.4f}")
    print(f"   • max   : {conf_np.max():.4f}")
    print(f"   • mean  : {conf_np.mean():.4f}")
    print(f"   • std   : {conf_np.std():.4f}\n")

    # ============================================================
    #   Select new pseudo-labels (balanced-ish per class)
    # ============================================================
    unlabeled_mask = (labels == -1)
    unlabeled_idx = torch.where(unlabeled_mask)[0]

    if unlabeled_idx.numel() == 0:
        print("No unlabeled samples left. Stopping curriculum.")
        break

    # high-confidence subset among unlabeled
    conf_unl = conf[unlabeled_idx]
    pred_unl = pred[unlabeled_idx]

    high_mask = conf_unl >= thr
    idx_high = unlabeled_idx[high_mask]

    if idx_high.numel() == 0:
        print("No new samples meet the threshold. Stopping curriculum.")
        break

    # sort high-conf by confidence (descending)
    conf_high = conf[idx_high]
    sort_idx = torch.argsort(conf_high, descending=True)
    idx_high = idx_high[sort_idx]
    pred_high = pred[idx_high]

    # split by predicted class
    idx_high_0 = idx_high[pred_high == 0]
    idx_high_1 = idx_high[pred_high == 1]

    n_pos = int(idx_high_1.numel())
    n_neg = int(idx_high_0.numel())

    # aturan: ambil semua high-conf pos,
    # dan batasi neg supaya nggak nge-flood
    max_neg_factor = 3   # max 3x jumlah pos
    min_neg_cap    = 50  # kalau nggak ada pos, ambil sampai 50 neg saja

    chosen_indices = []

    # selalu ambil semua high-conf positives
    if n_pos > 0:
        chosen_indices.append(idx_high_1)

        # berapa neg yang boleh diambil
        max_neg_allowed = min(n_neg, n_pos * max_neg_factor)
        if max_neg_allowed > 0:
            chosen_indices.append(idx_high_0[:max_neg_allowed])

    else:
        # kalau nggak ada pos sama sekali, tetap boleh nambah beberapa neg
        max_neg_allowed = min(n_neg, min_neg_cap)
        if max_neg_allowed > 0:
            chosen_indices.append(idx_high_0[:max_neg_allowed])

    if len(chosen_indices) == 0:
        print("High-confidence set exists but nothing selected after balancing. Stopping.")
        break

    chosen_indices = torch.cat(chosen_indices)
    new_mask = torch.zeros_like(labels, dtype=torch.bool)
    new_mask[chosen_indices] = True

    num_new = int(new_mask.sum())
    print(f"New pseudo-labeled samples added this round: {num_new}")
    print(f"   • new class-1 (fake) : {int((pred[new_mask] == 1).sum())}")
    print(f"   • new class-0 (real) : {int((pred[new_mask] == 0).sum())}")

    # Assign new pseudo-labels
    labels[new_mask] = pred[new_mask]

    # Updated labeled mask
    mask_labeled = (labels != -1)
    total_labeled = int(mask_labeled.sum())
    print(f"Total labeled samples after update: {total_labeled}\n")

    # Label distribution (after update)
    labeled_np = labels[mask_labeled].detach().cpu().numpy()
    unlabeled_count = (labels == -1).sum().item()
    unique_lbl, counts_lbl = np.unique(labeled_np, return_counts=True)

    print("Labeled class distribution:")
    for u, c in zip(unique_lbl, counts_lbl):
        print(f"   • Class {int(u)} : {int(c)} samples")
    print(f"   • Unlabeled (-1) : {unlabeled_count}")

    # --- Fine-tune ---
    print("Fine-tuning model...")
    epochs_round = 41
    for epoch in range(epochs_round):
        loss = train_epoch(mask_labeled)
        if epoch % 10 == 0 or epoch == epochs_round - 1:
            print(f"   [Round {r}] Epoch {epoch:02d} → loss = {loss:.4f}")

    print("\n--- End of Round", r, "---\n")




ROUND 1 — Confidence Threshold = 0.9900

 Confidence Statistics
   • min   : 0.5040
   • max   : 1.0000
   • mean  : 0.9837
   • std   : 0.0424

New pseudo-labeled samples added this round: 50
   • new class-1 (fake) : 0
   • new class-0 (real) : 50
Total labeled samples after update: 354

Labeled class distribution:
   • Class 0 : 348 samples
   • Class 1 : 6 samples
   • Unlabeled (-1) : 6107
Fine-tuning model...
   [Round 1] Epoch 00 → loss = 0.0344
   [Round 1] Epoch 10 → loss = 0.0291
   [Round 1] Epoch 20 → loss = 0.0240
   [Round 1] Epoch 30 → loss = 0.0193
   [Round 1] Epoch 40 → loss = 0.0153

--- End of Round 1 ---



ROUND 2 — Confidence Threshold = 0.9700

 Confidence Statistics
   • min   : 0.5001
   • max   : 1.0000
   • mean  : 0.9671
   • std   : 0.0873

New pseudo-labeled samples added this round: 24
   • new class-1 (fake) : 6
   • new class-0 (real) : 18
Total labeled samples after update: 378

Labeled class distribution:
   • Class 0 : 366 samples
   • Class 1 : 1

# Final Inference + Saves

After the curriculum finishes, we need to:

- Run one final forward pass.
- Extract the final probabilities (p_fake, p_real) and the final predictions.
- Add the following fields to df_hisn:

final_label <br>
final_p_fake / final_p_genuine <br>
label_source, with the following rules:
- "seed" → if the node came from the initial pseudo-labels (pseudo_label != -1)
- "self_train" → if it was originally -1 but received a label during curriculum
- "unlabeled" → if it is still -1 (never confident enough to be labeled_

In [108]:
model.eval()
with torch.no_grad():
    logits_final = model(data)
    probs_final = torch.softmax(logits_final, dim=1)
    conf_final, pred_final = probs_final.max(dim=1)

In [109]:
pred_np = pred_final.detach().cpu().numpy()
probs_np = probs_final.detach().cpu().numpy()
labels_np_final = labels.detach().cpu().numpy()

In [110]:
# Add to df_hisn
df_hisn["final_label"] = labels_np_final    # may still contain -1 for never-labeled
df_hisn["pred_label"]  = pred_np           # model's argmax prediction
df_hisn["p_genuine"]   = probs_np[:, 0]
df_hisn["p_fake"]      = probs_np[:, 1]


In [111]:
seed_mask = (df_hisn["pseudo_label"] != -1)
selftrain_mask = (df_hisn["pseudo_label"] == -1) & (df_hisn["final_label"] != -1)
unlabeled_mask_final = (df_hisn["final_label"] == -1)

In [112]:
label_source = np.full(len(df_hisn), "unlabeled", dtype=object)
label_source[seed_mask.to_numpy()] = "seed"
label_source[selftrain_mask.to_numpy()] = "self_train"

In [113]:
df_hisn["label_source"] = label_source

In [114]:
print("\nFinal label_source distribution:")
print(df_hisn["label_source"].value_counts())


Final label_source distribution:
label_source
self_train    6041
seed           304
unlabeled      116
Name: count, dtype: int64


In [115]:
df_hisn.to_csv(FINAL_PRED_CSV, index=False)
print("\n✅ Final predictions saved to", FINAL_PRED_CSV)


✅ Final predictions saved to nb5/final_predictions.csv


In [116]:
torch.save(model.state_dict(), MODEL_STATE_PATH)
print("✅ Model state_dict saved to", MODEL_STATE_PATH)

✅ Model state_dict saved to nb5/hgnn_model.pt


In [117]:
df_hisn.head(3)

,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,summary,...,orig_index,review_id,anomaly,user_suspicious,pseudo_label,final_label,pred_label,p_genuine,p_fake,label_source
0,5,fancy pumpkin headband,Purchased for my sister to use on Halloween......,B071HMN7K8,B071HMN7K8,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2021-10-27 04:30:00.822,0,True,,...,7,r_0,0.052410,0,-1,0,0,1.000000e+00,9.116234e-29,self_train
1,5,abalone swirl necklace,This is a large swirl abalone necklace... it i...,B01LZUP2XY,B01LZUP2XY,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2020-01-12 17:42:18.354,2,True,,...,8,r_1,0.046359,0,-1,0,0,1.000000e+00,8.028538e-33,self_train
2,5,white painting rocks,I purchased these for an art project with my 5...,B075Q24W9X,B075Q24W9X,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2019-05-22 03:04:51.057,1,True,,...,9,r_2,0.049485,0,-1,1,1,2.446099e-12,1.000000e+00,self_train


In [118]:
df_hisn["final_label"].value_counts()  # may still contain -1 for never-labeled

,count
final_label,
0,4635
1,1710
-1,116


In [119]:
df_hisn["pred_label"].value_counts()

,count
pred_label,
0,4697
1,1764
